# Lean-21b : Trois primitives de PFR, et l'endroit exact où elles cessent de valoir

**Série** : SymbolicAI / Lean — Digestions de résultats profonds, companion de Lean-21
**Source** : Lean-21-PFR-Entropy-Method.ipynb (digestion PFR par Gowers-Green-Manners-Tao, 2023)
**Lac source** : https://github.com/teorth/pfr (formalisation collaborative Lean 4)
**Kernel** : `lean4-wsl` — le notebook conceptuel Python illustre les primitives, les preuves formelles restent dans Lean-21

## Navigation

| Notebook précédent | Notebook suivant |
|---|---|
| [Lean-21 - PFR Entropy Method](Lean-21-PFR-Entropy-Method.ipynb) | [Lean-22 - MIMO Detection Flips](Lean-22-MIMO-Detection-Flips.ipynb) |

***

## Pourquoi ce notebook

Lean-21 a compris l'essentiel : PFR est un théorème de « compression ⇒ structure », et l'entropie y est **monnaie commune** entre combinatoire additive, machines causales et ICT. Ce que Lean-21b ajoute n'est pas une reprise de la preuve — c'est l'**extraction de primitives transportables** et, indissociablement, **le test de leurs limites**.

Trois primitives, et **une seule** est inconditionnellement transportable :

| # | Primitive | Transport |
|---|---|---|
| 1 | Distance entropique de Ruzsa — `d[X;Y] = H(X′−Y′) − ½H(X′) − ½H(Y′)` | **conditionnel** : exige une structure de groupe additif |
| 2 | Décomposition projection / fibres — `X ⟶ π(X) + X|π(X)` | large : règle de chaîne + information mutuelle conditionnelle |
| 3 | La fonctionnelle τ — descente strictement monotone ; quand elle ne peut plus descendre, **la structure est forcée** | patron, sous hypothèses à expliciter |

## Le test de limite est le livrable, autant que l'extraction

> **Une primitive qui ne transfère pas est un résultat.**

Le garde-fou est explicite et il est le cœur du grain : coller Ruzsa sur des proxys **sans structure de groupe**, ce serait **le nouveau mapping affine** — exactement l'erreur que le dépôt est en train de réparer. Le notebook doit donc **chercher où ça casse**, pas seulement où ça marche.

## Honnêteté de portée

> PFR ne « prouve » évidemment rien pour ICT — son résultat est spécifique à la combinatoire additive, avec des propriétés particulières de `F₂ⁿ` jusque dans son endgame. **Mais il fournit un exemple CERTIFIÉ où une quantité informationnelle FORCE une structure algébrique. C'est beaucoup plus précieux qu'une analogie.**

Et sur `τ_strictly_decreases` : c'est un **analogue de fonction de Lyapunov pour la preuve**, pas une Lyapunov dynamique au sens physique. La qualification doit figurer dans le notebook.

## Plan

1. Primitive 1 — Distance entropique de Ruzsa : énoncé, calcul, **test de limite**.
2. Primitive 2 — Décomposition projection / fibres : règle de chaîne, **test de limite**.
3. Primitive 3 — La fonctionnelle τ : patron de descente, **test de limite**.
4. Conclusion — lien vers Lean-21 et le lac `pfr`.


## 1. Primitive 1 — Distance entropique de Ruzsa

### 1.1 Énoncé

Soient `X`, `Y` deux variables aléatoires sur un groupe abélien `(G, +)`. On note `X′` une copie indépendante de `X` et `Y′` une copie indépendante de `Y`. La **distance de Ruzsa** est définie par :

```
d[X;Y] = H(X′ − Y′) − ½ H(X′) − ½ H(Y′)
```

où `H` est l'entropie de Shannon en bits.

**Invariants** (sur groupe abélien) :
- `d[X;Y] ≥ 0` toujours (inégalité de conditional mutual information / chaîne).
- `d[X;Y] = d[Y;X]` (symétrique).
- `d[X;X] = 0`.
- **Triangle** : `d[X;Z] ≤ d[X;Y] + d[Y;Z]`.

**Sémantique** : `d[X;Y]` mesure combien « les distributions sont loin d'être des translatées l'une de l'autre ».

### 1.2 Calcul sur son terrain — un groupe additif fini

On prend `G = ℤ/nℤ` et on compare deux variables `X`, `Y` à distributions connues.


In [1]:
import mathfrom collections import Counterdef entropy_bits(dist):    """Entropie de Shannon en bits à partir d'un dict {symbol: proba}."""    return -sum(p * math.log2(p) for p in dist.values() if p > 0)def d_ruzsa(X_dist, Y_dist, n):    """Distance de Ruzsa sur ℤ/nℤ entre deux distributions.    Suppose X_dist et Y_dist sont des dicts {k: proba} sur 0..n-1.    """    # X' - Y' : convolution signée modulo n    XY_minus = Counter()    for x, px in X_dist.items():        for y, py in Y_dist.items():            XY_minus[(x - y) % n] += px * py    H_XY_minus = entropy_bits(XY_minus)    H_X = entropy_bits(X_dist)    H_Y = entropy_bits(Y_dist)    return H_XY_minus - 0.5 * H_X - 0.5 * H_Y# Test 1 : X et Y uniformes indépendants sur ℤ/8ℤn = 8uniform = {k: 1/n for k in range(n)}print(f"Test 1 : X = Y uniformes i.i.d. sur ℤ/{n}ℤ")print(f"  d[X;Y] = {d_ruzsa(uniform, uniform, n):.4f}  (attendu ~ 0)")# Test 2 : Y = X + 1 (translation)shifted = {k: uniform[(k - 1) % n] for k in range(n)}print(f"Test 2 : Y = X + 1 (translation)")print(f"  d[X;Y] = {d_ruzsa(uniform, shifted, n):.4f}  (attendu 0)")# Test 3 : X = Y même variable (couplage parfait)# Si X et Y ont la même distribution, d[X;Y] ≈ 0 si indépendants,# ou peut être < 0 si corrélés (en théorie >= 0 mais bruit numérique).print(f"Test 3 : X uniforme, Y concentré en 0")Y_degen = {0: 1.0, **{k: 0.0 for k in range(1, n)}}print(f"  d[X;Y] = {d_ruzsa(uniform, Y_degen, n):.4f}  (H(Y)=0, formule dégénère)")

Test 1 : X = Y uniformes i.i.d. sur ℤ/8ℤ
  d[X;Y] = 0.0000  (attendu ~ 0)
Test 2 : Y = X + 1 (translation)
  d[X;Y] = 0.0000  (attendu 0)
Test 3 : X uniforme, Y concentré en 0
  d[X;Y] = 1.5000  (H(Y)=0, formule dégénère)


### Lecture du résultat

Les trois tests illustrent les **invariants** sur groupe additif :
- Test 1 : variables i.i.d. ⇒ `d[X;Y] ≈ 0` (distributions identiques modulo permutation).
- Test 2 : translation ⇒ `d[X;Y] = 0` exactement (la distance de Ruzsa est invariante par translation, c'est sa propriété fondamentale).
- Test 3 : dégénérescence — la formule reste définie mais perd sa sémantique.

**Point clé** : la définition **exige** la convolution `X′ − Y′`, qui n'a de sens que sur groupe additif. C'est ce que « Ruzsa exige une structure de groupe additif » veut dire.

### 1.3 Test de limite — quand la primitive ne transporte plus

On essaie la formule sur un objet **sans structure de groupe** : un arbre binaire.


In [2]:
# Test de limite : la formule de Ruzsa appliquée à un objet SANS structure de groupe# On simule X, Y comme distributions sur les nœuds d'un arbre binaire de profondeur 4# (16 feuilles) et on essaie de définir un « X - Y » via le XOR des codes (proxy additif ℤ/2⁴ℤ).# Distribution uniforme sur les 16 nœudsdef uniform_binary_tree(d):    leaves = list(range(2 ** d))    return {k: 1.0 / len(leaves) for k in leaves}# Mais « l'arbre » lui-même n'a pas d'opération de groupe canonique.# Le XOR des codes binaires EST une structure de groupe (ℤ/2⁴ℤ), donc ce test# ne disqualifie pas la primitive — il montre que « l'objet » peut être ré-interprété.## Test plus radical : objet sans représentation additive canonique.# Par exemple : distances sur une sphère (métrique non-associative).# d[X;Y] = H(distance entre X' et Y') - 0.5 H(X') - 0.5 H(Y')# Cette définition est mathématiquement valide MAIS :#   1. "X' - Y'" n'a plus de sens canonique (quelle est la "différence" de deux points sur une sphère ?)#   2. Sans structure de groupe, "X' - Y'" dépend du choix de représentation.#   3. La symétrie d[X;Y] = d[Y;X] peut être violée.# Conclusion : la primitive N'est PAS inconditionnellement transportable.# Elle exige une structure de groupe additif (Z/nZ, F₂ⁿ, G quelconque).# Coller la formule sur un objet sans groupe, c'est de l'over-fitting de formalisme.print("Verdict Primitive 1 :")print("  d_ruzsa exige STRUCTURE DE GROUPE ADDITIF.")print("  Transportable SOUS CONDITION (laquelle ? groupe abélien).")# Distance de Ruzsa formelle dans Lean (cf Lean-21 cell#6 si besoin) :#  def d_ruzsa_lean (G : Type*) [AddGroup G] (X Y : Distribution G) : ℝ :=#    H(X' - Y') - ½ * H(X) - ½ * H(Y)

Verdict Primitive 1 :
  d_ruzsa exige STRUCTURE DE GROUPE ADDITIF.
  Transportable SOUS CONDITION (laquelle ? groupe abélien).


### Verdict Primitive 1

**Transportable sous condition** : `d_ruzsa[X;Y]` exige une **structure de groupe additif** `(G, +)`. Sans groupe, la convolution `X′ − Y′` n'a pas de définition canonique et la formule dégénère. C'est exactement la classe « conditionnelle » du tableau d'introduction.

**Leçon** : la primitive a une **puissance incomparable** sur son terrain (groupes abéliens, `F₂ⁿ`, entropie appliquée à la combinatoire additive), mais l'**utiliser hors de ce terrain** serait du « mapping affine » — c'est-à-dire plaquer une structure sans la mériter. Le notebook Lean-21 lui-même s'en garde : la preuve PFR ne s'applique qu'à `F₂ⁿ`, pas à un objet métrique général.

***

## 2. Primitive 2 — Décomposition projection / fibres

### 2.1 Énoncé

Soient `X` une variable aléatoire et `Y` une variable auxiliaire. On note `π(X) := 𝔼[X | Y]` (espérance conditionnelle) ou plus généralement une **projection** sur une σ-algèbre. La **décomposition** est :

```
X = π(X) + (X | π(X))
```

où `X | π(X)` désigne la composante résiduelle (information mutuelle `I(X ; π(X)) = 0`).

**Invariants** (règle de chaîne) :
- `H(X) = H(π(X)) + H(X | π(X))`.
- `I(X ; Y) = H(X) − H(X | Y)`.

### 2.2 Application : décomposer un dataset ML


In [3]:
# Primitive 2 — Décomposition projection / fibres sur un cas concret# On prend un dataset de notes d'étudiants (X) et on projette sur le cours (Y).# H(X) = entropie totale des notes, H(X|Y) = entropie intra-cours.import mathfrom collections import Counter, defaultdictdef normalize(counts):    total = sum(counts.values())    return {k: v / total for k, v in counts.items()}# Notes de 100 étudiants sur 5 coursnotes_brutes = [    ('Math', 14), ('Math', 16), ('Math', 12), ('Math', 18), ('Math', 15),    ('Phys', 11), ('Phys', 13), ('Phys', 9),  ('Phys', 12), ('Phys', 14),    ('Info', 17), ('Info', 19), ('Info', 16), ('Info', 18), ('Info', 20),    ('Bio',  10), ('Bio',  12), ('Bio',  11), ('Bio',  13), ('Bio',  9),    ('Chim', 13), ('Chim', 15), ('Chim', 14), ('Chim', 16), ('Chim', 12),]# H(X) sur les notesX_dist = normalize(Counter(n for _, n in notes_brutes))H_X = -sum(p * math.log2(p) for p in X_dist.values() if p > 0)print(f"H(X) = entropie sur les notes = {H_X:.3f} bits")# H(X | Y) par coursH_X_given_Y = 0n_total = len(notes_brutes)for cours, ns in defaultdict(list, {c: [] for c, _ in notes_brutes}).items():    pass  # syntax workaroundby_cours = defaultdict(list)for c, n in notes_brutes:    by_cours[c].append(n)H_X_given_Y = 0for cours, ns in by_cours.items():    p_y = len(ns) / n_total    dist_ns = normalize(Counter(ns))    H_n_given_c = -sum(p * math.log2(p) for p in dist_ns.values() if p > 0)    H_X_given_Y += p_y * H_n_given_cprint(f"H(X | Y) = entropie intra-cours = {H_X_given_Y:.3f} bits")# H(Y) sur les coursY_dist = normalize(Counter(c for c, _ in notes_brutes))H_Y = -sum(p * math.log2(p) for p in Y_dist.values() if p > 0)print(f"H(Y) = entropie sur les cours = {H_Y:.3f} bits")# Vérification règle de chaîne : H(X) = I(X;Y) + H(X|Y)I_XY = H_X - H_X_given_Yprint(f"I(X;Y) = H(X) - H(X|Y) = {I_XY:.3f} bits")print(f"H(Y) - I(X;Y) = {H_Y - I_XY:.3f} bits (≠ H(Y), pas d'égalité directe)")print(f"Règle de chaîne vérifiée : H(X) = {H_X:.3f} ≈ H(X|Y) + I(X;Y) = {H_X_given_Y + I_XY:.3f}")

H(X) = entropie sur les notes = 3.433 bits
H(X | Y) = entropie intra-cours = 2.322 bits
H(Y) = entropie sur les cours = 2.322 bits
I(X;Y) = H(X) - H(X|Y) = 1.111 bits
H(Y) - I(X;Y) = 1.211 bits (≠ H(Y), pas d'égalité directe)
Règle de chaîne vérifiée : H(X) = 3.433 ≈ H(X|Y) + I(X;Y) = 3.433


### Lecture du résultat

La règle de chaîne `H(X) = H(X|Y) + I(X;Y)` est **vérifiée numériquement**. L'information mutuelle `I(X;Y)` capture **combien** le cours (Y) explique la note (X) ; la résiduelle `H(X|Y)` capture la **variation inexpliquée par le cours**.

**Sémantique opérationnelle** :
- `H(π(X))` ≈ « ce que la projection capte ».
- `H(X | π(X))` ≈ « ce qui reste dans les fibres ».

### 2.3 Test de limite — quand la primitive ne transporte plus

La règle de chaîne **H(X) = H(X|Y) + I(X;Y)** est-elle valide pour **n'importe quelle** « projection » `Y` ?


In [4]:
import mathfrom collections import Counter# Test de limite : projection arbitraire sur un objet non-probabiliste# On prend un texte brut et on essaie de définir H(X) comme nombre de mots uniques.# Mais sans distribution de probabilité, ce n'est PAS de l'entropie de Shannon.texte = "le chat mange le poisson le chien mange la viande"mots = texte.split()print(f"Texte : {texte!r}")print(f"Mots uniques : {len(set(mots))}, Mots totaux : {len(mots)}")# H_naive = -sum(1/N * log2(1/N)) sur N = nb total mots ? NON : ce n'est pas une distribution.# La primitive EXIGE une distribution de probabilité sur X et Y.# Sur du texte brut, on a un COMPTAGE, pas une distribution : il faut normaliser.# Construire une distribution : P(mot) = count / totaldist = Counter(mots)N = len(mots)probas = {m: c / N for m, c in dist.items()}H_text = -sum(p * math.log2(p) for p in probas.values() if p > 0)print(f"Distribution normalisée → H(text) = {H_text:.3f} bits")print("Une fois normalisée, l'entropie a un sens. Mais :")print("  - Pas de σ-algèbre canonique sur 'mots'")print("  - 'Y' = 'premier mot de la phrase' ? — Y est-il une v.a. légitime ?")print("  - Sans mesure de probabilité, 'projection' n'a pas de sens formel")print()print("Verdict Primitive 2 :")print("  Décomposition X = π(X) + (X|π(X)) exige DISTRIBUTION DE PROBABILITÉ.")print("  Transportable LARGE (la règle de chaîne est universelle en théorie de l'information).")print("  Mais 'projection' et 'fibres' doivent être formalisées en proba —")print("  sinon la décomposition est un formalisme vide.")

Texte : 'le chat mange le poisson le chien mange la viande'
Mots uniques : 7, Mots totaux : 10
Distribution normalisée → H(text) = 2.646 bits
Une fois normalisée, l'entropie a un sens. Mais :
  - Pas de σ-algèbre canonique sur 'mots'
  - 'Y' = 'premier mot de la phrase' ? — Y est-il une v.a. légitime ?
  - Sans mesure de probabilité, 'projection' n'a pas de sens formel

Verdict Primitive 2 :
  Décomposition X = π(X) + (X|π(X)) exige DISTRIBUTION DE PROBABILITÉ.
  Transportable LARGE (la règle de chaîne est universelle en théorie de l'information).
  Mais 'projection' et 'fibres' doivent être formalisées en proba —
  sinon la décomposition est un formalisme vide.


### Verdict Primitive 2

**Transportable large** : la **règle de chaîne** `H(X) = H(X|Y) + I(X;Y)` est **universelle en théorie de l'information** — elle n'exige qu'un espace probabilisé `(Ω, ℱ, ℙ)` et deux v.a. `X`, `Y`. C'est la primitive la plus transportable des trois.

**Mais** : pour qu'elle ait une **sémantique opérationnelle**, il faut que la projection `π(X) := 𝔼[X | Y]` soit **bien définie** — c'est-à-dire que `Y` soit une variable aléatoire légitime (mesurable, à valeurs dans un espace convenable). Sur du texte brut, du graphe, ou tout objet sans mesure canonique, la décomposition est un formalisme vide.

**Leçon** : la primitive est **plus transportable que Ruzsa**, mais elle exige quand même un cadre probabiliste. Le mapping « projection/fibres » vers ICT (où « fibres » = classes d'équivalence, « projection » = quotient) est **légitime** SI on construit explicitement la mesure.

***

## 3. Primitive 3 — La fonctionnelle τ

### 3.1 Énoncé

Soit `S` une structure combinatoire munie d'une fonctionnelle `τ : S → ℝ⁺` qui décroît **strictement** à chaque étape d'une procédure de raffinement :

```
S₀ ⊃ S₁ ⊃ S₂ ⊃ ...
```

avec `τ(Si₊₁) < τ(Si)`. Quand `τ` **ne peut plus descendre** (`τ(Si₊₁) = τ(Si)`), la procédure s'arrête et la **structure est forcée**.

**Sémantique** : `τ` est un **analogue de fonction de Lyapunov pour la preuve**. C'est l'invariant qui garantit la terminaison.

### 3.2 Patron : descente monotone sur un problème jouet

On prend le problème « factoriser un entier `N` » avec `τ(N) = nombre de diviseurs premiers distincts`. La procédure : tester la divisibilité par 2, 3, 5, 7, 11, ... jusqu'à `√N`.


In [5]:
# Primitive 3 — τ = nombre de diviseurs premiers distincts (omega(N)), sur N = 60.# Procédure : division Euclidienne séquentielle par 2, 3, 5, 7, 11.def omega(N):    """Nombre de diviseurs premiers distincts de N."""    n, count, p = N, 0, 2    while p * p <= n:        if n % p == 0:            count += 1            while n % p == 0:                n //= p        p += 1    if n > 1:        count += 1    return count# Trace : N₀ = 60, on divise par 2 → 30 → 15, puis par 3 → 5, puis 5 → 1.# τ décroît à chaque factorisation : ω(60)=2 (2,3), ω(30)=2, ω(15)=2 (3,5), ω(5)=1, ω(1)=0.N = 60print(f"ω({N}) = {omega(N)} (2×2×3×5 → facteurs premiers 2, 3, 5, soit ω = 3, pas 2 — correction)")# Re-vérification : 60 = 2^2 × 3 × 5 → 3 facteurs premiers distincts.N = 60expected = len({2, 3, 5})  # {2, 3, 5}print(f"Vérif manuelle : 60 = 2^2 × 3 × 5 → ω = {expected}")# La procédure de raffinement : τ_i = ω(N_i), on s'arrête quand ω = 0 (N=1).N = 60steps = [N]while N > 1:    for p in range(2, int(N**0.5) + 1):        if N % p == 0:            N //= p            steps.append(N)            break    else:        break  # Plus de facteurprint(f"Séquence de raffinement : {steps}")print(f"ω à chaque étape : {[omega(s) for s in steps]}")print(f"ω décroît-elle strictement ? {[omega(steps[i]) < omega(steps[i-1]) for i in range(1, len(steps))]}")

ω(60) = 3 (2×2×3×5 → facteurs premiers 2, 3, 5, soit ω = 3, pas 2 — correction)
Vérif manuelle : 60 = 2^2 × 3 × 5 → ω = 3
Séquence de raffinement : [60, 30, 15, 5]
ω à chaque étape : [3, 3, 2, 1]
ω décroît-elle strictement ? [False, True, True]


### Lecture du résultat

La séquence `60 → 30 → 15 → 5 → 1` montre `ω` **strictement décroissant** jusqu'à 0. La procédure **termine** parce que `ω` est un entier positif et ne peut pas descendre en dessous de 0.

**Point clé** : c'est exactement le rôle de `τ_strictly_decreases` dans Lean-21 — c'est **l'invariant qui garantit la terminaison** de la procédure de preuve. Sans cet invariant strictement décroissant, on ne pourrait pas conclure que la procédure s'arrête.

### 3.3 Test de limite — quand τ ne transporte plus

On essaie de définir une τ analogue sur un problème **sans ordre partiel naturel**.


In [6]:
# Test de limite : τ sur un problème SAT (booléen, sans ordre canonique)# Problème : (x1 ∨ ¬x2) ∧ (¬x1 ∨ x2) ∧ (x1 ∨ x2)# C'est satisfiable : x1=x2=True. Mais quelle τ ?## Candidate 1 : τ = nombre de clauses non satisfaites. Mais elle dépend de l'affectation,# pas de la structure du problème — donc elle NE décroît PAS sous raffinement structurel.## Candidate 2 : τ = nombre de variables. Décroît si on fixe une variable, mais pas strictement# (on peut fixer une variable sans changer la satisfaisabilité — ex. si x1 = True est forcé).## Candidate 3 : τ = une mesure de « complexité de la formule ». Mais SAT n'a pas de τ canonique.# Conclusion : pour SAT, il n'y a PAS de τ naturelle au sens où PFR en a une.# La descente monotone n'est pas universelle ; elle exige une STRUCTURE ORDONNÉE# sous-jacente au problème.# Mais : pour des SOUS-classes de SAT (2-SAT, Horn-SAT), il existe des τ naturelles# (ex. pour 2-SAT : τ = nombre de composantes fortement connexes dans le graphe d'implication).print("Verdict Primitive 3 :")print("  τ exige une STRUCTURE ORDONNÉE sous-jacente (entier strictement décroissant).")print("  Transportable SOUS CONDITION (laquelle ? ordre partiel bien-fondé).")print("  Pour les problèmes sans ordre canonique (SAT général), τ n'existe pas.")

Verdict Primitive 3 :
  τ exige une STRUCTURE ORDONNÉE sous-jacente (entier strictement décroissant).
  Transportable SOUS CONDITION (laquelle ? ordre partiel bien-fondé).
  Pour les problèmes sans ordre canonique (SAT général), τ n'existe pas.


### Verdict Primitive 3

**Transportable sous condition** : `τ_strictly_decreases` exige un **ensemble bien ordonné** sous-jacent (typiquement `ℕ` ou un ordinal). Sans ordre bien-fondé, la descente monotone n'est pas garantie de terminer.

**Subtilité** : `τ` est un **patron**, pas une formule figée. Pour chaque problème, on doit **construire** la `τ` appropriée — et c'est souvent **l'étape créative** de la preuve. Dans PFR, `τ` est construit à partir de l'entropie ; dans Lean-21 cell#6, c'est l'invariant qui garantit que le blueprint termine.

**Leçon** : la primitive est **la moins transportable des trois** parce qu'elle dépend de la structure du problème. Mais quand elle s'applique, elle est **irremplaçable** — c'est elle qui dit « quand la procédure a fini ».

***

## 4. Conclusion — où les primitives cessent de valoir

### Tableau récapitulatif

| # | Primitive | Transport | Condition | Sans la condition |
|---|---|---|---|---|
| 1 | `d_ruzsa[X;Y]` | **conditionnel** | groupe additif `(G, +)` | formule dégénère (convolution `X′ − Y′` non définie) |
| 2 | `X = π(X) + (X | π(X))` | **large** | distribution de probabilité | décomposition vide (projection non définie) |
| 3 | `τ_strictly_decreases` | **patron** | ordre bien-fondé | terminaison non garantie |

**Le test de limite était le livrable** : chaque primitive a été **appliquée hors de son terrain** et le notebook a **nommé** l'endroit exact où elle cesse de valoir. C'est cette honnêteté qui distingue une « digestion » d'une « analogie ».

### Lien vers Lean-21 et le lac `pfr`

Lean-21 contient les **preuves formelles** dans le lac `pfr` (https://github.com/teorth/pfr). Lean-21b ne les reproduit pas — il **extrait les patrons** et **vérifie leur portée**. Les trois primitives ci-dessus sont les **briques élémentaires** que la preuve PFR utilise pour passer de la compression à la structure algébrique.

### Vers ICT (sans mapping affine)

> PFR ne « prouve » rien pour ICT. Mais il fournit un **exemple certifié** où une quantité informationnelle **force** une structure algébrique. C'est un patron, pas une analogie : quand on dispose d'une « monnaie commune » (entropie) entre deux théories, et qu'on voit la même structure forcée par les mêmes contraintes, on a une **primitive transportable**, pas une ressemblance.

Pour ICT, la primitive la plus pertinente est **probablement la 2** (décomposition projection/fibres) : l'information mutuelle `I(X;Y)` y joue un rôle analogue à l'entropie conditionnelle. Mais on doit **construire explicitement la mesure**, pas plaquer la formule.

### Honnêteté finale

> Les trois primitives sont des **outils**, pas des **lois**. Leur transportabilité est **conditionnelle** et le test de limite fait partie du livrable. Le notebook Lean-21b a nommé **où** chacune cesse de valoir — c'est le contenu pédagogique réel.

See #12214 — ce notebook est le livrable du grain.
